In [1]:
# !pip install openai
# !pip install git+https://github.com/huggingface/transformers.git
# !pip install accelerate
# !pip install pdfplumber
# !pip install PyMuPDF
# !pip install torch
# !pip install sentencepiece
# !pip install protobuf

In [2]:
import requests
import torch
import fitz
import io
import base64
from PIL import Image
from IPython.display import display
from openai import OpenAI
from typing import List, Optional
from pydantic import BaseModel, Field

from prompts import (
    LANGUAGE_MODIFIER,
    LENGTH_MODIFIERS,
    QUESTION_MODIFIER,
    SYSTEM_PROMPT,
    TONE_MODIFIER,
)

In [3]:
PDF_URL = "https://arxiv.org/pdf/2501.16634"
PDF_PATH = "paper.pdf"

LLM_MODEL = "google/gemma-3-27b-it"
LLM_URL = "http://localhost:8000/v1"

In [4]:
def download_pdf(url: str, output_path: str):
    response = requests.get(url)
    if response.status_code == 200:
        with open(output_path, "wb") as f:
            f.write(response.content)
    else:
        raise Exception(f"Failed to download PDF, status code: {response.status_code}")

In [5]:
def page_to_image(page):
    # Render the full page to a pixmap (as RGB image)
    zoom = 2  # Increase resolution (1 = 72 DPI, 2 = 144 DPI)
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    # Convert to PIL image
    image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return image

In [6]:
def encode_image(image):
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    encoded = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"

In [7]:
class Dialogue(BaseModel):
    character: str = Field(
        ..., description="Name of the character delivering the dialogue."
    )
    transcript: str = Field(
        ..., description="Transcript of the dialogue. Must not be more than 50 words."
    )

    def __str__(self):
        str_ = f"{self.character}: {self.transcript}"
        return str_

class Script(BaseModel):
    dialogues: List[Dialogue] = Field(
        ..., description="Ordered list of dialogues in the script."
    )

    def __str__(self):
        str_ = "Script:\n\n"
        for dialogue in self.dialogues:
            str_ += f"{dialogue.__str__()}\n\n"
        return str_

class Scene(BaseModel):
    characters: List[str] = Field(
        ..., description="List of characters shown in this scene."
    )
    dialogues: List[Dialogue] = Field(
        ..., description="Ordered list of dialogues in the scene."
    )

    def __str__(self):
        str_ = f"Characters: {self.characters}\n\n"
        str_ += "Transcript:\n\n"
        for dialogue in self.dialogues:
            str_ += f"{dialogue.__str__()}\n\n"
        return str_

class Podcast(BaseModel):
    scenes: List[Scene] = Field(
        ..., description="Ordered list of scenes in the podcast."
    )

    def __str__(self):
        str_ = ""
        for scene_idx, scene in enumerate(self.scenes):
            str_ += f"Scene: {scene_idx}\n{scene.__str__()}"
        return str_

In [8]:
download_pdf(PDF_URL, PDF_PATH)

In [9]:
doc = fitz.open(PDF_PATH)

pdf_text = []
pdf_images = []
for pg_num, page in enumerate(doc):
    text = page.get_text()
    images = page.get_images(full=True)
    print(f"Page: {pg_num}, Text: {len(text)}")
    print(f"Page: {pg_num}, Images: {len(images)}")
    pdf_text.append(text)
    
    for img_index, img in enumerate(images):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        encoded_image = encode_image(image)
        pdf_images.append(encoded_image)

Page: 0, Text: 4966
Page: 0, Images: 9
Page: 1, Text: 5315
Page: 1, Images: 0
Page: 2, Text: 5063
Page: 2, Images: 0
Page: 3, Text: 5364
Page: 3, Images: 0
Page: 4, Text: 4765
Page: 4, Images: 0
Page: 5, Text: 7107
Page: 5, Images: 0
Page: 6, Text: 2050
Page: 6, Images: 0


In [10]:
for pg_num, page in enumerate(doc):
    image = page_to_image(page)
    encoded_image = encode_image(image)
    pdf_images.append(encoded_image)

In [11]:
client = OpenAI(api_key="n/a", base_url=LLM_URL)

In [12]:
question = "What is the main theme of this paper?"

# Prepare the user prompt text string
user_prompt_text = "\n\n".join(pdf_text)
user_prompt_text += f"\n\n{QUESTION_MODIFIER} {question}"

# Add images to the user prompt
user_prompt = [{"type": "text", "text": user_prompt_text}]
for image_url in pdf_images:
    user_prompt.append({"type": "image_url", "image_url": {"url": image_url}})

# Prepare the system prompt text string
system_prompt = SYSTEM_PROMPT
system_prompt += f"\n\n{QUESTION_MODIFIER} {question}"

# Combine the prompts into a single message
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = client.beta.chat.completions.parse(
    model=LLM_MODEL,
    messages=messages,
    temperature=0.6,
    max_tokens=5000,
    extra_body=dict(guided_decoding_backend="xgrammar"),
    response_format=Script,
)

In [14]:
response_message = response.choices[0].message
if response_message.parsed:
    script = response_message.parsed

In [15]:
print(script)

Script:

Jane: Hello and welcome to the show! Today, we’re diving into a fascinating paper titled 'Towards Resource-Efficient Compound AI Systems.'  I’m thrilled to have with us someone who can break down this complex topic – let’s welcome Gohar Irfan Chaudhry, one of the authors! Gohar, welcome to the podcast.

Gohar: Hi Jane, thanks so much for having me. I’m happy to be here.

Jane: So, Gohar, for our listeners who might not be deeply familiar with the world of AI systems, could you start by giving us a high-level overview? What *is* a Compound AI System, and why are they becoming so important?

Gohar: Absolutely.  Essentially, a Compound AI System is a system that tackles complex tasks by combining multiple interacting components. Think of it like building with LEGOs. Instead of one giant, monolithic AI model, you're connecting different specialized pieces – things like different AI models, tools for retrieving information, or even external services. These systems are becoming esse

In [16]:
# Prepare the user prompt text string
user_prompt = "Given the podcast description, split the dialogues into scenes such that some scenes focus on one character, others on multiple characters."
user_prompt += f"\n\nThe transcript is as follows: {script}"

# Prepare the system prompt text string
system_prompt = "You are a world-class podcast director who can organize a script into a collection of scenes that will be later converted into a video."

# Combine the prompts into a single message
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [17]:
response = client.beta.chat.completions.parse(
    model=LLM_MODEL,
    messages=messages,
    temperature=0.6,
    max_tokens=5000,
    extra_body=dict(guided_decoding_backend="xgrammar"),
    response_format=Podcast,
)

In [18]:
response_message = response.choices[0].message
if response_message.parsed:
    podcast = response_message.parsed

In [19]:
print(podcast)

Scene: 0
Characters: ['Jane', 'Gohar']

Transcript:

Jane: Hello and welcome to the show! Today, we’re diving into a fascinating paper titled 'Towards Resource-Efficient Compound AI Systems.'  I’m thrilled to have with us someone who can break down this complex topic – let’s welcome Gohar Irfan Chaudhry, one of the authors! Gohar, welcome to the podcast.

Gohar: Hi Jane, thanks so much for having me. I’m happy to be here.

Jane: So, Gohar, for our listeners who might not be deeply familiar with the world of AI systems, could you start by giving us a high-level overview? What *is* a Compound AI System, and why are they becoming so important?

Gohar: Absolutely.  Essentially, a Compound AI System is a system that tackles complex tasks by combining multiple interacting components. Think of it like building with LEGOs. Instead of one giant, monolithic AI model, you're connecting different specialized pieces – things like different AI models, tools for retrieving information, or even extern